In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy qiskit-addon-utils
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

<details>
<summary><b>Package versions</b></summary>

The code on this page was developed using the following requirements.
We recommend using these versions or newer.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
qiskit-addon-utils~=0.3.0
```
</details>

חבילת כלי העזר של תוספי Qiskit היא אוסף של פונקציונליות להשלמת תהליכי עבודה הכוללים תוסף אחד או יותר של Qiskit. לדוגמה, החבילה מכילה פונקציות ליצירת המילטוניאנים, יצירת מעגלי אבולוציית זמן מסוג Trotter, ופריסה ושילוב של מעגלים קוונטיים.

## התקנה

ישנן שתי דרכים להתקין את כלי העזר של תוספי Qiskit: מ-PyPI ובנייה מקוד המקור. מומלץ להתקין חבילות אלו ב[סביבה וירטואלית](https://docs.python.org/3.10/tutorial/venv.html) כדי להבטיח הפרדה בין תלויות החבילות.

### התקנה מ-PyPI

הדרך הפשוטה ביותר להתקין את חבילת כלי העזר של תוספי Qiskit היא דרך PyPI.

In [1]:
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke
from qiskit import QuantumCircuit
from qiskit_addon_utils.coloring import auto_color_edges
from qiskit_addon_utils.slicing import combine_slices, slice_by_depth
from collections import defaultdict

backend = FakeSherbrooke()
coupling_map = backend.coupling_map

# Create naive circuit
circuit = QuantumCircuit(backend.num_qubits)
for edge in coupling_map.graph.edge_list():
    circuit.cz(edge[0], edge[1])


# Color the edges of the coupling map
coloring = auto_color_edges(coupling_map)
circuit_with_coloring = QuantumCircuit(backend.num_qubits)

# Make a reverse coloring dict in order to make the circuit
color_to_edge = defaultdict(list)
for edge, color in coloring.items():
    color_to_edge[color].append(edge)

# Place edges in order of color
for edges in color_to_edge.values():
    for edge in edges:
        circuit_with_coloring.cz(edge[0], edge[1])

print(f"The circuit without using edge coloring has depth: {circuit.depth()}")
print(
    f"The circuit using edge coloring has depth: {circuit_with_coloring.depth()}"
)

The circuit without using edge coloring has depth: 37
The circuit using edge coloring has depth: 3


### התקנה מקוד המקור
<details>
<summary>
לחץ כאן לקרוא כיצד להתקין את החבילה הזו ידנית.
</summary>

אם ברצונך לתרום לחבילה זו או להתקין אותה ידנית, שכפל תחילה את המאגר:

In [2]:
import numpy as np
from qiskit import QuantumCircuit

num_qubits = 9
qc = QuantumCircuit(num_qubits)
qc.ry(np.pi / 4, range(num_qubits))
qubits_1 = [i for i in range(num_qubits) if i % 2 == 0]
qubits_2 = [i for i in range(num_qubits) if i % 2 == 1]
qc.cx(qubits_1[:-1], qubits_2)
qc.cx(qubits_2, qubits_1[1:])
qc.cx(qubits_1[-1], qubits_1[0])
qc.rx(np.pi / 4, range(num_qubits))
qc.rz(np.pi / 4, range(num_qubits))
qc.draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/d5a5c53f-45d2-4cdc-86f3-221f179cdbea-0.svg" alt="Output of the previous code cell" />

והתקן את החבילה דרך `pip`. אם אתה מתכנן להריץ את הדרכות שנמצאות במאגר החבילה, התקן גם את תלויות ה-notebook. אם אתה מתכנן לפתח במאגר, התקן את תלויות `dev`.

In [3]:
# Slice circuit into partitions of depth 1
slices = slice_by_depth(qc, 1)

# Recombine slices in order to visualize the partitions together
combined_slices = combine_slices(slices, include_barriers=True)
combined_slices.draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/6d824d97-edc6-4c88-bcfa-428731393f83-0.svg" alt="Output of the previous code cell" />

</details>

## תחילת עבודה עם כלי העזר
ישנם מספר מודולים בחבילת `qiskit-addon-utils`, כולל אחד ליצירת בעיות לסימולציית מערכות קוונטיות, צביעת גרפים להצבת שערים במעגל קוונטי בצורה יעילה יותר, ופריסת מעגלים, שיכולה לסייע עם [החזרה לאחור של אופרטורים](/guides/qiskit-addons-obp). הסעיפים הבאים מסכמים כל מודול. [תיעוד ה-API](https://docs.quantum.ibm.com/api/qiskit-addon-utils) של החבילה מכיל גם הוא מידע שימושי.

### יצירת בעיות
תכולת המודול [`qiskit_addon_utils.problem_generators`](../api/qiskit-addon-utils/problem-generators) כוללת:

- פונקציית [`generate_xyz_hamiltonian()`](../api/qiskit-addon-utils/problem-generators#generate_xyz_hamiltonian) שמייצרת ייצוג `SparsePauliOp` המודע לקישוריות של מודל XYZ מסוג איזינג:

$$H = \sum_{(j,k)\in E} \left(J_x X_jX_k + J_yY_jY_k + J_zZ_jZ_k\right) + \sum_{j\in V} \left(h_x X_j + h_y Y_j + h_z Z_j\right) $$
- פונקציית [`generate_time_evolution_circuit()`](../api/qiskit-addon-utils/problem-generators#generate_time_evolution_circuit) שבונה מעגל המדמה את אבולוציית הזמן של אופרטור נתון.
- שלושה אובייקטי [`PauliOrderStrategy`](../api/qiskit-addon-utils/problem-generators#pauliorderstrategy) שונים לאיטרציה בין סדרי מחרוזות Pauli שונים. שימושי בעיקר בשימוש לצד צביעת גרפים, וניתן להשתמש בהם כארגומנטים גם בפונקציות `generate_xyz_hamiltonian()` וגם ב-`generate_time_evolution_circuit()`.

### צביעת גרפים
המודול [`qiskit_addon_utils.coloring`](../api/qiskit-addon-utils/coloring) משמש לצביעת הצלעות במפת הצימוד ולשימוש בצביעה זו כדי להציב שערים במעגל קוונטי בצורה יעילה יותר. מטרת מפת הצימוד הצבועה בצלעות היא למצוא קבוצת צבעי צלעות כך שאין שתי צלעות באותו הצבע שחולקות צומת משותפת. עבור QPU, פירוש הדבר שניתן להריץ שערים לאורך צלעות בעלות צבע זהה (חיבורי Qubit) בו-זמנית, והמעגל יבוצע מהר יותר.

כדוגמה מהירה, ניתן להשתמש בפונקציית [`auto_color_edges()`](../api/qiskit-addon-utils/coloring#auto_color_edges) כדי ליצור צביעת צלעות עבור מעגל נאיבי שמבצע `CZGate` לאורך כל חיבור Qubit. קטע הקוד שלהלן משתמש במפת הצימוד של ה-backend של `FakeSherbrooke`, יוצר מעגל נאיבי זה, ולאחר מכן משתמש בפונקציית `auto_color_edges()` כדי ליצור מעגל שקול יעיל יותר.

In [4]:
from qiskit_addon_utils.slicing import slice_by_gate_types

slices = slice_by_gate_types(qc)

# Recombine slices in order to visualize the partitions together
combined_slices = combine_slices(slices, include_barriers=True)
combined_slices.draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/d011c56e-d741-491a-8867-54952cb7b757-0.svg" alt="Output of the previous code cell" />

If your workload is designed to exploit the physical qubit connectivity of the QPU it will be run on, you can create slices based on edge coloring. The code snippet below will assign a three-coloring to circuit edges and slice the circuit with respect to the edge coloring. (Note: this only affects non-local gates. Single-qubit gates will be sliced by gate type).

In [5]:
from qiskit_addon_utils.slicing import slice_by_coloring

# Assign a color to each set of connected qubits
coloring = {}
for i in range(num_qubits - 1):
    coloring[(i, i + 1)] = i % 3
coloring[(num_qubits - 1, 0)] = 2

# Create a circuit with operations added in order of color
qc = QuantumCircuit(num_qubits)
qc.ry(np.pi / 4, range(num_qubits))
edges = [
    edge for color in range(3) for edge in coloring if coloring[edge] == color
]
for edge in edges:
    qc.cx(edge[0], edge[1])
qc.rx(np.pi / 4, range(num_qubits))
qc.rz(np.pi / 4, range(num_qubits))

# Create slices by edge color
slices = slice_by_coloring(qc, coloring=coloring)

# Recombine slices in order to visualize the partitions together
combined_slices = combine_slices(slices, include_barriers=True)
combined_slices.draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/41290d70-e116-4bd2-b9e7-d90aeaa852aa-0.svg" alt="Output of the previous code cell" />

### פריסה
לבסוף, המודול [`qiskit-addon-utils.slicing`](../api/qiskit-addon-utils/slicing) מכיל פונקציות ומעברי Transpiler לעבודה עם יצירת "פרוסות" מעגל, חלוקות דמויות-זמן של [`QuantumCircuit`](../api/qiskit/qiskit.circuit.QuantumCircuit) המשתרעות על פני כל ה-Qubits. פרוסות אלו משמשות בעיקר ל[החזרה לאחור של אופרטורים](/guides/qiskit-addons-obp-get-started). ארבע הדרכים העיקריות לפריסת מעגל הן לפי סוג שער, עומק, צביעה, או הוראות [`Barrier`](../api/qiskit/circuit#barrier). הפלט של פונקציות הפריסה הללו מחזיר רשימה של אובייקטי [`QuantumCircuit`](../api/qiskit/qiskit.circuit.QuantumCircuit). ניתן גם לשלב מחדש מעגלים פרוסים באמצעות פונקציית [`combine_slices()`](../api/qiskit-addon-utils/slicing#combine_slices). קרא את [מסמך API](../api/qiskit-addon-utils/slicing) של המודול לקבלת מידע נוסף.

להלן מספר דוגמאות כיצד ליצור פרוסות אלו באמצעות המעגל הבא:

In [6]:
qc = QuantumCircuit(num_qubits)
qc.ry(np.pi / 4, range(num_qubits))
qc.barrier()
qubits_1 = [i for i in range(num_qubits) if i % 2 == 0]
qubits_2 = [i for i in range(num_qubits) if i % 2 == 1]
qc.cx(qubits_1[:-1], qubits_2)
qc.cx(qubits_2, qubits_1[1:])
qc.cx(qubits_1[-1], qubits_1[0])
qc.barrier()
qc.rx(np.pi / 4, range(num_qubits))
qc.rz(np.pi / 4, range(num_qubits))
qc.draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/5040a678-9d5f-4c8b-8a92-7de04c3fcfda-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/qiskit-addons-utils/extracted-outputs/d5a5c53f-45d2-4cdc-86f3-221f179cdbea-0.svg)

במקרה שאין דרך ברורה לנצל את מבנה המעגל להחזרה לאחור של אופרטורים, ניתן לחלק את המעגל לפרוסות בעלות עומק נתון.

In [7]:
from qiskit_addon_utils.slicing import slice_by_barriers

slices = slice_by_barriers(qc)
slices[0].draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/b1106c2f-711c-4d30-b91a-ea05f47598d8-0.svg" alt="Output of the previous code cell" />

In [8]:
slices[1].draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/1afd2111-dd88-4e20-a137-f0c975e9d1bb-0.svg" alt="Output of the previous code cell" />

In [9]:
slices[2].draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/360ab773-df81-4975-bb19-cd5f34e69b29-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/qiskit-addons-utils/extracted-outputs/41290d70-e116-4bd2-b9e7-d90aeaa852aa-0.svg)

אם יש לך אסטרטגיית פריסה מותאמת אישית, ניתן במקום זאת להציב מחסומים במעגל כדי לתחום היכן הוא יפורס ולהשתמש בפונקציית [`slice_by_barriers`](../api/qiskit-addon-utils/slicing#slice_by_barriers).